# Team agents: subagent steps in the progress display

A team agent hands work to its subagents through `delegate_task`. Each `delegate_task` step now carries:

- `execution_id`: the subagent's own platform execution
- `steps`: the subagent's own steps, nested under the delegation, updated live while it runs

The engine side shipped in aixplain-agents [#439](https://github.com/aixplain/aixplain-agents/pull/439). The SDK side, which renders it in `agent.run(..., progress_format="logs" | "status")`, is [aixplain/aiXplain#1153](https://github.com/aixplain/aiXplain/pull/1153), branch `feature/ENG-3708-sdk-nested-delegation-progress`.

This notebook builds a real team (a web researcher with Tavily Web Search, a haiku writer, and a coordinator) and runs it. It needs only an aiXplain API key. The engine change is deployed on **dev**, so the client below points there.

In [1]:
# Install the SDK from the PR branch (dependencies first, then this branch's build over any installed release).
%pip install -q "aiXplain @ git+https://github.com/aixplain/aiXplain.git@feature/ENG-3708-sdk-nested-delegation-progress"
%pip install -q --force-reinstall --no-deps "aiXplain @ git+https://github.com/aixplain/aiXplain.git@feature/ENG-3708-sdk-nested-delegation-progress"


If the import below still points at an older version, restart the kernel once after the install.

In [2]:
from importlib.metadata import version

from aixplain.v2 import agent_progress

assert hasattr(agent_progress, "_is_delegation"), "this SDK build predates nested progress: restart the kernel and rerun"
print("aiXplain SDK", version("aiXplain"), "with nested delegation progress")


aiXplain SDK 0.3.0 with nested delegation progress


## Client

The API key comes from `AIXPLAIN_API_KEY`, or you are prompted for it. The hosts default to dev, where the engine change runs. Override them with `BACKEND_URL` / `MODELS_RUN_URL`.

In [3]:
import os
from getpass import getpass

from aixplain import Aixplain

API_KEY = os.getenv("AIXPLAIN_API_KEY") or getpass("aiXplain API key: ")
BACKEND_URL = os.getenv("BACKEND_URL", "https://dev-platform-api.aixplain.com")
MODELS_RUN_URL = os.getenv("MODELS_RUN_URL", "https://dev-models.aixplain.com/api/v2/execute")

aix = Aixplain(api_key=API_KEY, backend_url=BACKEND_URL, model_url=MODELS_RUN_URL)
print("Connected to", BACKEND_URL)


Connected to https://dev-platform-api.aixplain.com


## Build the team

Two subagents and a coordinator. Each is looked up by name first, so re-running the notebook reuses them instead of creating duplicates. The coordinator's instructions name the subagents exactly: `delegate_task` routes by name.

In [4]:
LLM = "669a63646eb56306647e1091"           # GPT-4o Mini
TAVILY = "6931bdf462eb386b7158def3"        # Tavily Web Search


def find_agent(name):
    """Exact name match over your agents (a query search is fuzzy and can miss fresh agents)."""
    page_number = 0
    while True:
        page = aix.Agent.search(page_size=100, page_number=page_number)
        match = next((a for a in page.results if a.name == name), None)
        page_number += 1
        if match or not page.results or page_number * 100 >= (page.total or 0):
            return match


def get_or_create(name, **fields):
    found = find_agent(name)
    if found:
        return aix.Agent.get(found.id)
    agent = aix.Agent(name=name, llm=LLM, **fields)
    agent.save()
    return agent


researcher = get_or_create(
    "Nested Progress Demo - Web Researcher",
    description="Searches the web for current news and returns 3 sourced facts.",
    instructions="ROLE: Web researcher. CONSTRAINTS: Always call Tavily Web Search before answering; never answer "
                 "from memory. OUTPUT RULES: 3 bullet points, each ending with its source URL.",
    tools=[aix.Tool.get(TAVILY)],
)
haiku_writer = get_or_create(
    "Nested Progress Demo - Haiku Writer",
    description="Writes one haiku about a topic.",
    instructions="Write a single haiku (3 lines) about the topic you are given. Output only the haiku.",
)
team = get_or_create(
    "Nested Progress Demo - News Team",
    description="Coordinates a web researcher and a haiku writer.",
    instructions=f"For every topic, delegate IN THE SAME TURN to the agent named '{researcher.name}' (current news) "
                 f"and to the agent named '{haiku_writer.name}' (a haiku). Then combine both into one answer.",
    subagents=[researcher, haiku_writer],
)
for agent in (researcher, haiku_writer, team):
    print(f"{agent.name:<40} {agent.id}")


Nested Progress Demo - Web Researcher    6abe5b441552c29b0b0f6735
Nested Progress Demo - Haiku Writer      6abe5b455f16f5aef518795a
Nested Progress Demo - News Team         6abe5b465f16f5aef518795b


## Run it: `progress_format="logs"`

Each delegation prints a `▸` header, then its subagent's own steps (`2.1`, `2.2`, …) as they finish, then its ✓ line. Here the web researcher's steps include the Tavily call. The two subagents run in parallel, and one line at the bottom stays live: the step running right now. The completion total sums the top-level steps, because a delegation carries its subagent's total cost.

In [5]:
QUERY = "Latest news about the Mediterranean Sea"

result = team.run(QUERY, progress_format="logs", timeout=600)
print()
print(result.data.output)


✓ Step  1 · ⏱ 00:00.00 · API  1 · $0.000174 · ↓835 ↑82 (917) · Nested Progress Demo - News Team ⧈ GPT-4o Mini
▸ Step  2 · Nested Progress Demo - News Team ⚒ delegate_task → Nested Progress Demo - Web Researcher
▸ Step  3 · Nested Progress Demo - News Team ⚒ delegate_task → Nested Progress Demo - Haiku Writer
  ✓ Step 3.1 · ⏱ 00:00.00 · API  1 · $0.000123 · ↓745 ↑18 (763) · Nested Progress Demo - Haiku Writer ⧈ GPT-4o Mini
✓ Step  3 · ⏱ 00:04.15 · API  1 · $0.000147 · Nested Progress Demo - News Team ⚒ delegate_task → Nested Progress Demo - Haiku Writer
  ✓ Step 2.1 · ⏱ 00:00.00 · API  1 · $0.000137 · ↓847 ↑17 (864) · Nested Progress Demo - Web Researcher ⧈ GPT-4o Mini
  ✓ Step 2.2 · ⏱ 00:00.00 · API  1 · $0.000750 · Nested Progress Demo - Web Researcher ⚒ Tavily Web Search
  ✓ Step 2.3 · ⏱ 00:03.55 · API  1 · $0.000515 · ↓2672 ↑190 (2862) · Nested Progress Demo - Web Researcher ⧈ GPT-4o Mini
✓ Step  2 · ⏱ 00:13.66 · API  1 · $0.001682 · Nested Progress Demo - News Team ⚒ delegate_task 

## `progress_verbosity=2`: subagent execution ids

From verbosity 2, the delegation header also shows the subagent's platform `execution_id`. You can poll or open that execution on its own.

In [6]:
result = team.run(QUERY, progress_format="logs", progress_verbosity=2, timeout=600)


✓ Step  1 · ⏱ 00:04.78 · API  1 · $0.000174 · ↓835 ↑82 (917) · Nested Progress Demo - News Team ⧈ GPT-4o Mini
▸ Step  2 · Nested Progress Demo - News Team ⚒ delegate_task → Nested Progress Demo - Web Researcher · ↳ execution 949-f9f59c6d-dc18-4851-a95b-c204a26e9b8a
▸ Step  3 · Nested Progress Demo - News Team ⚒ delegate_task → Nested Progress Demo - Haiku Writer · ↳ execution 949-b59ea269-b75e-4134-93bb-fd77b4ba72d8
  ✓ Step 3.1 · ⏱ 00:00.00 · API  1 · $0.000123 · ↓745 ↑18 (763) · Nested Progress Demo - Haiku Writer ⧈ GPT-4o Mini
✓ Step  3 · ⏱ 00:02.38 · API  1 · $0.000147 · Nested Progress Demo - News Team ⚒ delegate_task → Nested Progress Demo - Haiku Writer
  ✓ Step 2.1 · ⏱ 00:00.00 · API  1 · $0.000137 · ↓847 ↑17 (864) · Nested Progress Demo - Web Researcher ⧈ GPT-4o Mini
  ✓ Step 2.2 · ⏱ 00:00.00 · API  1 · $0.000750 · Nested Progress Demo - Web Researcher ⚒ Tavily Web Search
  ✓ Step 2.3 · ⏱ 00:01.58 · API  1 · $0.000518 · ↓2672 ↑195 (2867) · Nested Progress Demo - Web Researcher

## `progress_format="status"`: one updating line

The single line follows a running delegation down to the step its subagent is on, e.g. `Step 2.2 · … ⚒ Tavily Web Search`. In a saved notebook you only see its final state; run the cell to watch it move.

In [7]:
result = team.run(QUERY, progress_format="status", timeout=600)


✓ Step  4 · ⏱ 00:02.14 · API  1 · $0.000298 · ↓1149 ↑210 (1359) · Nested Progress Demo - News Team ⧈ GPT-4o Mini
✓ Completed 4 steps · ⏱ 00:19.35 · API 4 · $0.002302 · ↓1984 ↑292 (2276)


## The data behind it

The display reads `result.data.steps`. A `delegate_task` step holds its subagent's `execution_id` and `steps`.

In [8]:
def show(steps, depth=0):
    for step in steps:
        unit, agent = step.get("unit") or {}, (step.get("agent") or {}).get("name")
        extra = f"  execution_id={step['execution_id']}" if step.get("execution_id") else ""
        print(f"{'    ' * depth}- [{step.get('status')}] {unit.get('type')}: {unit.get('name')} ({agent})"
              f"  credits={step.get('used_credits')}{extra}")
        show(step.get("steps") or [], depth + 1)


show(result.data.steps)


- [completed] model: GPT-4o Mini (Nested Progress Demo - News Team)  credits=0.00017444999999999998
- [completed] tool: delegate_task (Nested Progress Demo - News Team)  credits=0.0016817400000000001  execution_id=949-e3c309de-d6eb-47ba-a327-12e695dacf47
    - [completed] model: GPT-4o Mini (Nested Progress Demo - Web Researcher)  credits=0.00013725
    - [completed] tool: Tavily Web Search (Nested Progress Demo - Web Researcher)  credits=0.00075
    - [completed] model: GPT-4o Mini (Nested Progress Demo - Web Researcher)  credits=0.0005142
- [completed] tool: delegate_task (Nested Progress Demo - News Team)  credits=0.00014706  execution_id=949-508292b0-1eaf-454a-97b1-b85e882a7438
    - [completed] model: GPT-4o Mini (Nested Progress Demo - Haiku Writer)  credits=0.00012255
- [completed] model: GPT-4o Mini (Nested Progress Demo - News Team)  credits=0.00029834999999999996
